# Field Generation

To create a target-based pharmacophore,
at the very least a set of 3D scalar fields are needed,
each measuring some metric for the favorability of a pharamacophore feature
at different points within the binding pocket of a receptor.
The modeler then provides various methods
to extract pharmacophore features from the corresponding field values.

In [1]:
import numpy as np     # To generate mock data
import t2fpharm_study  # To obtain a ready-to-use chemical system
import t2fpharm

## From Pre-Computed Data

The shortest and most flexible way to create a `Field` object
is by providing pre-computed field data to the `t2fpharm.field.from_tensor()` function.
This requires providing spatial information
defining the grid on which the fields are sampled,
as well as the corresponding array containing the field values.
For example:

In [2]:
# A 20x30x40 grid starting at coordinates (0, 0, 0) with 0.3 Å spacing
GRID_ORIGIN = (0, 0, 0)
GRID_SHAPE = (20, 30, 40)
GRID_SPACING = 0.3

# Field values for 3 feature metrics measured for a single receptor instance.
FEATURE_IDS = ("hbond_acceptor", "hbond_donor", "hydrophobic")
FIELD_TENSOR = np.ones((len(FEATURE_IDS), *GRID_SHAPE))
# You can also provide a field tensor that measures the same feature metrics
# for multiple receptor instances (i.e., multiple conformations, e.g., from MD simulations).
# In that case, the tensor must have the shape (n_features, n_instances, *grid_shape).

In [3]:
# Create the grid
field_grid: t2fpharm.grid.Grid = t2fpharm.grid.from_anchor_shape_spacing(
    anchor=GRID_ORIGIN,
    shape=GRID_SHAPE,
    spacing=GRID_SPACING,
    anchor_type="lower",
)
# Create the field
field_from_data: t2fpharm.field.Field = t2fpharm.field.from_tensor(
    tensor=FIELD_TENSOR,
    grid=field_grid,
    feature_ids=FEATURE_IDS
)

A `Grid` object can also be generated from various other input data.
For more information, see the notebook [`4_grid.ipynb`](./4_grid.ipynb).

## From AutoGrid

Fields can be generated using AutoDock AutoGrid.
For this, a receptor structure is required as a PDBQT file.
Here, we use a pre-generated PDBQT file for brevity;
for more information on creating a PDBQT file, see the notebook [`2_system.ipynb`](./2_system.ipynb).

In [4]:
receptor_pdbqt: str = t2fpharm_study.manager().pdbqt("1aq1")

We can now create the fields using the `t2fpharm.field.from_autogrid()` function,
providing the AutoGrid ligand types to generate fields for.
Note that here we are using the same `Grid` created above;
if you have a pocket, you can instead directly use the `pocket.Grid` object instead:

In [5]:
# Create fields
field_from_ligand: t2fpharm.field.Field = t2fpharm.field.from_autogrid(
    grid=field_grid,
    receptor_files=receptor_pdbqt,
    ligand_types=("HD", "C", "OA", "e+", "e-"),
)

Again, here you can also provide multiple receptor files,
each for a different conformation of the receptor.
More information can be found in the function's docstring:

In [6]:
help(t2fpharm.field.from_autogrid)

Help on function from_autogrid in module t2fpharm.field:

from_autogrid(grid: 'Grid', receptor_files: 'str | bytes | Path | ArrayLike', ligand_types: 'Sequence[str]' = ('HD', 'C', 'OA', 'e-', 'e+'), receptor_types: 'Sequence[str] | None' = None, identical_receptor_types: 'bool' = True, smooth: 'float' = 0.5, dielectric: 'float' = -0.1465, parameter_files: 'str | bytes | Path | ArrayLike | None' = None, receptor_file_ids: 'str | Sequence[tuple[str, Sequence[str]]] | None' = None, parameter_file_ids: 'str | Sequence[tuple[str, Sequence[str]]] | None' = None, field_dtype: 'DTypeLike' = <class 'jax.numpy.float32'>, output_dir: 'str | Path' = None, allow_copy: 'bool' = True) -> 'Field'
    Create a field using AutoDock AutoGrid4.

    Parameters
    ----------
    grid
        A `Grid` object containing the grid information.
        The grid must be a 3D orthogonal grid
        with equal spacing in all dimensions.
        However, in contrast to working directly with AutoGrid,
        here

## Field Methods and Attributes

The `Field` object has two main attributes: `grid` and `tensor`.
The `grid` attribute returns a `Grid` object, discussed in the notebook [`4_grid.ipynb`](./4_grid.ipynb).
The `tensor` attributes returns a `JAX.Array` object, containing the field values:

In [7]:
field_from_ligand.grid

Grid(
  shape=[20 30 40],
  size=[ 5.7  8.7 11.7],
  spacings=[0.3 0.3 0.3],
  lower_bounds=[0. 0. 0.],
  center=[2.85 4.35 5.85],
  upper_bounds=[ 5.7  8.7 11.7]
)

In [8]:
field_from_ligand.tensor.shape

(5, 20, 30, 40)

The first dimension of the tensor corresponds to different fields,
each corresponding to a feature type.
The middle dimensions (if any) correspond to batch dimensions
each representing an instance of the field (e.g., for different receptor conformations),
while the last three dimensions match the grid shape.
For each feature type, the corresponding field(s) can also be accessed by the given feature ID. For example:

In [9]:
field_from_ligand(feature="HD").shape

(20, 30, 40)